# Local Evaluation - Groundedness

After you have setup and configured the prompt flow, its time to evaluation its performance. Here we can use the prompt flow SDK to test different questions and see how the prompt flow performs using the evaluation prompt flows provided.

In [2]:
from promptflow import PFClient

pf_client = PFClient()

from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Add a question to test the base prompt flow.
question = "Can you tell me about your jackets?"
customerId = "4"
output = pf_client.test(
    flow="../contoso-chat", # Path to the flow directory
    inputs={ # Inputs to the flow
        "chat_history": [],
        "question": question,
        "customerId": customerId,
    },
)

output["answer"] = "".join(list(output["answer"]))

[2024-04-15 19:39:15,471][promptflow._sdk.entities._connection][WARNING] - Please use connection.secrets[key] to access secrets.


2024-04-15 19:39:16 +0000    1387 execution.flow     INFO     Start executing nodes in thread pool mode.
2024-04-15 19:39:16 +0000    1387 execution.flow     INFO     Start to run 5 nodes with concurrency level 16.
2024-04-15 19:39:16 +0000    1387 execution.flow     INFO     Executing node question_embedding. node run id: b06e2978-67e7-47ab-a701-029216ae167e_question_embedding_0
2024-04-15 19:39:16 +0000    1387 execution.flow     INFO     Executing node customer_lookup. node run id: b06e2978-67e7-47ab-a701-029216ae167e_customer_lookup_0
2024-04-15 19:39:17 +0000    1387 execution.flow     INFO     Node customer_lookup completes.
2024-04-15 19:39:17 +0000    1387 execution.flow     INFO     Node question_embedding completes.
2024-04-15 19:39:17 +0000    1387 execution.flow     INFO     Executing node retrieve_documentation. node run id: b06e2978-67e7-47ab-a701-029216ae167e_retrieve_documentation_0
2024-04-15 19:39:18 +0000    1387 execution.flow     INFO     Node retrieve_documentatio

In [4]:
output

{'answer': "Hey Sarah Lee! 🌟\nWe have two awesome jackets for outdoor adventures 🏞️:\n1. Summit Breeze Jacket (Product ID: 3): Lightweight, windproof, water-resistant, and has reflective accents for visibility.\n2. RainGuard Hiking Jacket (Product ID: 17): Waterproof, breathable, and equipped with ventilation zippers for increased airflow. Perfect for evening treks.\nBoth jackets are perfect for your outdoor outings and would go great with the CozyNights Sleeping Bag and the TrailBlaze Hiking Pants you've purchased before! 🎒🏕️",
 'context': [{'id': '3',
   'title': 'Summit Breeze Jacket',
   'content': "Discover the joy of hiking with MountainStyle's Summit Breeze Jacket. This lightweight jacket is your perfect companion for outdoor adventures. Sporting a trail-ready, windproof design and a water-resistant fabric, it's ready to withstand any weather. The breathable polyester material and adjustable cuffs keep you comfortable, whether you're ascending a mountain or strolling through a p

Test the groundedness of the prompt flow with the answer from the above question.

In [5]:
test = pf_client.test(
    flow="groundedness",
    inputs={
        "question": question,
        "context": str(output["context"]),
        "answer": output["answer"],
    },
)

2024-04-15 19:39:21 +0000    1387 execution.flow     INFO     Start executing nodes in thread pool mode.
2024-04-15 19:39:21 +0000    1387 execution.flow     INFO     Start to run 2 nodes with concurrency level 16.
2024-04-15 19:39:21 +0000    1387 execution.flow     INFO     Executing node groundedness_score. node run id: f50e8c9a-b1de-4206-af4d-0df17bdd2c0d_groundedness_score_0
2024-04-15 19:39:21 +0000    1387 execution.flow     INFO     Node groundedness_score completes.
2024-04-15 19:39:21 +0000    1387 execution.flow     INFO     Executing node concat_scores. node run id: f50e8c9a-b1de-4206-af4d-0df17bdd2c0d_concat_scores_0
2024-04-15 19:39:21 +0000    1387 execution.flow     INFO     Node concat_scores completes.
2024-04-15 19:39:21 +0000    1387 execution.flow     INFO     Start to run 1 nodes with concurrency level 16.
2024-04-15 19:39:21 +0000    1387 execution.flow     INFO     Executing node aggregate_variants_results. node run id: 93723fc9-8424-4416-989e-b0d178e7a41a_aggre

In [6]:
test

{'gpt_groundedness': 5.0}

# Local Evaluation - Multiple Metrics 

Now use the same prompt flow and test it against the Multi Evaluation flow for groundedness, coherence, fluency, and relevance.

In [7]:
test_multi = pf_client.test(
    "multi_flow",
    inputs={
        "question": question,
        "context": str(output["context"]),
        "answer": output["answer"],
    },
)

2024-04-15 19:39:27 +0000    1387 execution.flow     INFO     Start executing nodes in thread pool mode.
2024-04-15 19:39:27 +0000    1387 execution.flow     INFO     Start to run 8 nodes with concurrency level 16.
2024-04-15 19:39:27 +0000    1387 execution.flow     INFO     Executing node coherence_score. node run id: b34945f1-35ca-4492-8c48-067ca1367207_coherence_score_0
2024-04-15 19:39:27 +0000    1387 execution.flow     INFO     Executing node fluency_score. node run id: b34945f1-35ca-4492-8c48-067ca1367207_fluency_score_0
2024-04-15 19:39:27 +0000    1387 execution.flow     INFO     Executing node groundedness_score. node run id: b34945f1-35ca-4492-8c48-067ca1367207_groundedness_score_0
2024-04-15 19:39:27 +0000    1387 execution.flow     INFO     Executing node relevance_score. node run id: b34945f1-35ca-4492-8c48-067ca1367207_relevance_score_0
2024-04-15 19:39:27 +0000    1387 execution.flow     INFO     Node groundedness_score completes.
2024-04-15 19:39:27 +0000    1387 exec

In [8]:
test_multi

{'gpt_coherence': 4.0,
 'gpt_fluency': 5.0,
 'gpt_groundedness': 5.0,
 'gpt_relevance': 3.0}

# AI Studio Azure batch run on an evaluation json dataset

Now in order to test these more thoroughly, we can use the Azure AI Studio to run batches of test data with the evaluation prompt flow on a larger dataset.

In [9]:
import json
# Import required libraries
from promptflow.azure import PFClient

# Import required libraries
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

In [10]:
try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()

Populate the `config.json` file with the subscription_id, resource_group, and workspace_name.

In [11]:
config_path = "../config.json"
pf_azure_client = PFClient.from_config(credential=credential, path=config_path)

Found the config file in: ../config.json


Add the runtime from the AI Studio that will be used for the cloud batch runs.

In [12]:
# Update the runtime to the name of the runtime you created previously
runtime = "automatic"
# load flow
flow = "../contoso-chat"
# load data
data = "../data/salestestdata.jsonl"

In [13]:
# get current time stamp for run name
import datetime
now = datetime.datetime.now()
timestamp = now.strftime("%Y_%m_%d_%H%M%S")
run_name = timestamp+"chat_base_run"
print(run_name)

2024_04_15_193949chat_base_run


Create a base run to use as the variant for the evaluation runs. 

_NOTE: If you get "'An existing connection was forcibly closed by the remote host'" run the cell again._

In [14]:
# create base run in Azure Ai Studio
base_run = pf_azure_client.run(
    flow=flow,
    data=data,
    column_mapping={
        # reference data
        "customerId": "${data.customerId}",
        "question": "${data.question}",
    },
    runtime=runtime,
    # create a display name as current datetime
    display_name=run_name,
    name=run_name
)
print(base_run)

Uploading contoso-chat (0.68 MBs): 100%|██████████| 679466/679466 [00:00<00:00, 2860341.51it/s]


[2024-04-15 19:39:51,098][promptflow][WARNING] - You're using automatic runtime, if it's first time you're using it, it may take a while to build runtime and you may see 'NotStarted' status for a while. 


Portal url: https://ai.azure.com/projectflows/bulkrun/run/2024_04_15_193949chat_base_run/details?wsid=/subscriptions/fc0be99c-c8bc-44d9-b98a-7775c82fd517/resourcegroups/rg-contoso-chat-ai/providers/Microsoft.MachineLearningServices/workspaces/contoso-chat-ai-project
name: 2024_04_15_193949chat_base_run
created_on: '2024-04-15T19:39:54.910266+00:00'
status: NotStarted
display_name: 2024_04_15_193949chat_base_run
description:
tags: {}
properties:
  azureml.promptflow.runtime_name: automatic
  azureml.promptflow.inputs_mapping: '{"customerId":"${data.customerId}","question":"${data.question}"}'
  azureml.promptflow.disable_trace: 'false'
  azureml.promptflow.session_id: e63fe9a4ec7de4081a5c5faf44b4a1969952d9ddf5b5eba3
  azureml.promptflow.definition_file_name: flow.dag.yaml
  azureml.promptflow.flow_lineage_id: a6117409b8b717c7e79942d8573769a6d1f4ac180e7fabad302590d2b43bff3c
  azureml.promptflow.flow_definition_datastore_name: workspaceblobstore
  azureml.promptflow.flow_definition_blob_p

In [15]:
pf_azure_client.stream(base_run)

(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
======= Run Summary =======
Run name: "2024_04_15_193949chat_base_run"
Run status: "Failed"
Start time: "2024-04-15 19:44:04.469978+00:00"
Duration: "0:00:00"
Run url: "https://ai.azure.com/projectflows/bulkrun/run/2024_04_15_193949chat_base_run/details?wsid=/subscriptions/fc0be99c-c8bc-44d9-b98a-7775c82fd517/resourcegroups/rg-contoso-chat-ai/providers/Microsoft.MachineLearningServices/workspaces/contoso-chat-ai-project"

InvalidRunStatusError: Failed to submit bulk run due to Not found connection aoai-connection for user Brad Stevens

In [ ]:
details = pf_azure_client.get_details(base_run)
details.head(10)

# Cloud Eval run on Json Data

In [ ]:
eval_flow = "multi_flow/"
data = "../data/salestestdata.jsonl"
run_name = timestamp+"chat_eval_run"
print(run_name)

eval_run_variant = pf_azure_client.run(
    flow=eval_flow,
    data=data,  # path to the data file
    run=base_run,  # use run as the variant
    column_mapping={
        # reference data
        "customerId": "${data.customerId}",
        "question": "${data.question}",
        "context": "${run.outputs.context}",
        # reference the run's output
        "answer": "${run.outputs.answer}",
    },
    runtime=runtime,
    display_name=run_name,
    name=run_name
)

In [ ]:
pf_azure_client.stream(eval_run_variant)

In [ ]:
details = pf_azure_client.get_details(eval_run_variant)
details.head(10)

In [ ]:

metrics = pf_azure_client.get_metrics(eval_run_variant)
print(json.dumps(metrics, indent=4))

In [ ]:
pf_azure_client.visualize([base_run, eval_run_variant])